# AlgoCourse Lecture 2: Finding an Alpha
**Imperial Algorithmic Trading Society**

Tonight we follow one trade, **Shell against BP**, from the idea to managing it. There are five short tasks, each matching a **Notebook task** slide in the lecture.

**How to use this notebook**
- Run the setup cell first (click it, then Shift + Enter). It loads the data from the course GitHub.
- In each task, replace every `___` with the right code, then run the cell.
- Each task lists the functions you'll need. If you get stuck, we show the solution after each task.
- To keep your work, use **File → Save a copy in Drive**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, acf

URL = "https://raw.githubusercontent.com/ekaterinaveter/AlgoCourse-2026/main/shell_bp.csv"
prices = pd.read_csv(URL, index_col="Date", parse_dates=True).dropna()   # daily closing prices, pence
prices.tail()

## Task 1: prices or returns?
**Why:** statistics only works on a series whose average and size of moves stay the same over time (stationarity). Check which of prices and returns behaves like that.

1. Compute daily log returns, $r_t = \ln(P_t / P_{t-1})$.
2. For each year: the average price, the average daily return, and the annualised volatility.
3. Look at the plots. Which one stays stable from year to year?

**Functions you'll need**
| What | Code |
|---|---|
| Log of every price | `np.log(prices)` |
| Change from the day before | `.diff()` |
| Drop the first, empty row | `.dropna()` |
| Average for each year | `df.groupby(df.index.year).mean()` |
| Standard deviation for each year | `df.groupby(df.index.year).std()` |
| Square root | `np.sqrt(252)` |

In [ ]:
# 1. Daily log returns
returns = np.log(prices).diff().dropna()

# 2. Year by year
avg_price  = prices.groupby(prices.index.year).mean()
avg_return = 100 * returns.groupby(returns.index.year).mean()
volatility = 100 * returns.groupby(returns.index.year).std() * np.sqrt(252)

print("Average price (pence):");          print(avg_price.round(0))
print("\nAverage daily return (%):");     print(avg_return.round(2))
print("\nAnnualised volatility (%):");    print(volatility.round(1))

# 3. Plots (already written for you)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(100 * prices / prices.iloc[0]).plot(ax=ax[0], title="Prices, rebased to 100 at the start")
(100 * returns).plot(ax=ax[1], title="Daily returns (%)", lw=0.5, alpha=0.6)
plt.show()

## Task 2: autocorrelation
**Why:** a trading signal needs the past to predict the future. Autocorrelation $\rho(k)$ is the correlation between a series and its own value $k$ days earlier.

Using 2015 to 2019, compute $\rho(1)$ to $\rho(10)$ for:
1. Shell's daily returns
2. BP's daily returns
3. The spread, $\ln(\text{Shell}) - \ln(\text{BP})$

Which one can you predict from its past?

**Functions you'll need**
| What | Code |
|---|---|
| $\rho(1)$ to $\rho(10)$ of a series | `acf(series, nlags=10)[1:]` |
| Shell's returns, 2015 to 2019 | `r["SHEL"]` |
| The spread, 2015 to 2019 | `spread` |

In [ ]:
r = returns.loc[:"2019"]
logp = np.log(prices).loc[:"2019"]
spread = logp["SHEL"] - logp["BP"]

acf_shell  = acf(r["SHEL"], nlags=10)[1:]
acf_bp     = acf(r["BP"], nlags=10)[1:]
acf_spread = acf(spread, nlags=10)[1:]

# Plot (already written for you)
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, values, title in zip(ax, [acf_shell, acf_bp, acf_spread], ["Shell returns", "BP returns", "Spread"]):
    a.bar(range(1, 11), values); a.axhline(0, color="grey"); a.set_title(title); a.set_xlabel("lag k")
plt.show()
print("rho(1): Shell", acf_shell[0].round(2), " BP", acf_bp[0].round(2), " spread", acf_spread[0].round(2))

## Task 3: build and test the spread
**Why:** to trade the spread we need (1) the hedge ratio β, (2) a test that the spread is stationary, and (3) how fast it comes back. Fit everything on 2015 to 2019.

1. Regress log Shell on log BP. The slope is the hedge ratio β.
2. Build the spread $s_t = \ln \text{Shell}_t - c - \beta \ln \text{BP}_t$ for the whole period, and run the Engle-Granger test on 2015 to 2019.
3. Half-life: regress $\Delta s_t$ on $s_{t-1}$. With slope $\gamma$, the half-life is $-\ln 2 / \ln(1 + \gamma)$ days.
4. z-score: $z_t = (s_t - \text{mean}) / \text{std}$, with the mean and std from 2015 to 2019.

**Functions you'll need**
| What | Code |
|---|---|
| Regression of y on x with a constant | `sm.OLS(y, sm.add_constant(x)).fit()` |
| Its constant and slope | `model.params["const"]`, `model.params["BP"]` |
| Engle-Granger test | `coint(y, x)` returns (statistic, p-value, critical values) |
| Yesterday's value of a series | `s.shift(1)` |

In [ ]:
logp = np.log(prices)
fit = logp.loc[:"2019"]                                   # the data we fit on

# 1. Hedge ratio
model = sm.OLS(fit["SHEL"], sm.add_constant(fit["BP"])).fit()
beta = model.params["BP"]
print(f"Hedge ratio beta: {beta:.2f}")

# 2. Spread for the whole period, and the Engle-Granger test on 2015 to 2019
spread = logp["SHEL"] - model.params["const"] - beta * logp["BP"]
stat, pvalue, crit = coint(fit["SHEL"], fit["BP"])
print(f"Engle-Granger statistic {stat:.2f}, 5% critical value {crit[1]:.2f}, p-value {pvalue:.3f}")

# 3. Half-life
s = spread.loc[:"2019"]
hl_model = sm.OLS(s.diff().dropna(), sm.add_constant(s.shift(1).dropna())).fit()
gamma = hl_model.params.iloc[1]
print(f"Half-life: {-np.log(2) / np.log(1 + gamma):.0f} trading days")

# 4. z-score, using the 2015 to 2019 mean and std
z = (spread - s.mean()) / s.std()
z.loc[:"2019"].plot(figsize=(10, 4), title="z-score of the spread, 2015 to 2019")
for level in (-2, 2):
    plt.axhline(level, color="red", linestyle="--")
plt.axhline(0, color="grey"); plt.show()

## Task 4: backtest the rule on 2015 to 2019
**Why:** a backtest runs the rule on past data to see what it would have made.

The rule: if $z > +2$, sell the spread (position $-1$). If $z < -2$, buy it ($+1$). Close when $z$ crosses back through 0.

1. Fill in the rule in `positions`.
2. Daily P&L = **yesterday's** position × today's change in the spread. (Why yesterday's?)
3. Run the backtest and read the Sharpe ratio, maximum drawdown and number of trades.

**Functions you'll need**
| What | Code |
|---|---|
| Yesterday's position | `pos.shift(1)` |
| Today's change in the spread | `spread.diff()` |
| Annualised Sharpe ratio | `ret.mean() / ret.std() * np.sqrt(252)` |

In [ ]:
def positions(z, entry=2.0, exit=0.0):
    pos, current = [], 0
    for zi in z:
        if current == 0 and zi > entry:
            current = -1           # sell the spread
        elif current == 0 and zi < -entry:
            current = 1            # buy the spread
        elif current == 1 and zi >= exit:
            current = 0
        elif current == -1 and zi <= exit:
            current = 0
        pos.append(current)
    return pd.Series(pos, index=z.index)


def backtest(z, spread, cost_bp=5.0, plot=True):
    pos = positions(z)
    pnl = (pos.shift(1) * spread.diff()).fillna(0)             # yesterday's position x today's change
    costs = pos.diff().abs().fillna(0) * (1 + beta) * cost_bp / 1e4
    ret = (pnl - costs) / (1 + beta)                           # return on the money in both legs
    cum = ret.cumsum()
    if plot:
        (100 * cum).plot(figsize=(10, 4), title="Cumulative return (%)"); plt.show()
    return {"Sharpe": round(float(ret.mean() / ret.std() * np.sqrt(252)), 2),
            "max drawdown": f"{(cum - cum.cummax()).min():.1%}",
            "trades": int(((pos != 0) & (pos.shift(1).fillna(0) == 0)).sum())}


backtest(z.loc[:"2019"], spread.loc[:"2019"])

## Task 5: what happened next?
**Why:** a backtest on the data you fitted on can't tell you whether the rule works on new data. Run the **same** rule, with the same β, mean and std, on 2020 to today. Don't refit anything.

1. Backtest 2020 onwards.
2. Plot the z-score for the whole period.
3. Rerun the Engle-Granger test on 2020 onwards. What changed?

**Functions you'll need**
| What | Code |
|---|---|
| Data from 2020 onwards | `.loc["2020":]` |
| The backtest from Task 4 | `backtest(z, spread)` |
| Engle-Granger test | `coint(y, x)` |

In [ ]:
# 1. Same rule, data it has never seen
print(backtest(z.loc["2020":], spread.loc["2020":]))

# 2. z-score for the whole period (already written for you)
ax = z.loc[:"2019"].plot(figsize=(10, 4), label="fitted: 2015 to 2019")
z.loc["2020":].plot(ax=ax, color="red", label="traded: 2020 onwards")
for level in (-2, 2):
    ax.axhline(level, color="grey", linestyle="--")
ax.legend(); plt.show()

# 3. Engle-Granger on 2020 onwards
stat, pvalue, crit = coint(logp.loc["2020":, "SHEL"], logp.loc["2020":, "BP"])
print(f"2020 onwards: statistic {stat:.2f}, 5% critical value {crit[1]:.2f}, p-value {pvalue:.2f}")
print(f"z today: {z.iloc[-1]:.1f}")